In [ ]:
import os
import math
import json
import time
import torch
import shutil
import zipfile
import requests
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets
from torch.nn import functional as F
from torchvision.io import read_image
from torchvision import transforms as T
from torch.utils.data import DataLoader, Dataset
torch.cuda.manual_seed(42)

In [2]:
transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor()
])

dataset = datasets.FakeData(transform=transform)

def benchmark_num_workers(batch_size=64, num_workers_list=[0, 2, 4, 8]):
    results = {}
    for nw in num_workers_list:
        loader = DataLoader(dataset,
                            batch_size=batch_size,
                            shuffle=True,
                            num_workers=nw,
                            pin_memory=True)
        
        start = time.time()
        # Run through one full epoch
        for batch in loader:
            # Simulate GPU transfer
            batch = batch[0].to("cuda", non_blocking=True)
        end = time.time()
        
        epoch_time = end - start
        results[nw] = epoch_time
        print(f"num_workers={nw} → epoch_time={epoch_time:.2f}s")
    return results

if __name__ == "__main__":
    results = benchmark_num_workers()
    print("\nSummary:", results)

num_workers=0 → epoch_time=3.55s
num_workers=2 → epoch_time=1.68s
num_workers=4 → epoch_time=1.38s


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


num_workers=8 → epoch_time=1.44s

Summary: {0: 3.5453760623931885, 2: 1.6766698360443115, 4: 1.3751380443572998, 8: 1.4394583702087402}


In [3]:
size = 64
patch = 16
dropout = 0.3
n_layers = 12
num_workers = 4
batch_size = 128
max_iterations = 80
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
torch.cuda.device_count()

2

In [5]:
!nvidia-smi

Fri Feb 27 05:01:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             27W /   70W |     125MiB /  15360MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
r = requests.get(url, timeout=10)
r.raise_for_status()

with open('tiny-net.zip', 'wb') as f:
  f.write(r.content)

In [7]:
zip_path = 'tiny-net.zip'
extraction_path = './tiny-net'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall(extraction_path)

In [8]:
os.listdir(os.path.join(os.path.join(extraction_path,'tiny-imagenet-200'), 'train'))[:10]

['n02509815',
 'n07720875',
 'n02883205',
 'n01644900',
 'n03388043',
 'n03837869',
 'n02437312',
 'n01770393',
 'n04328186',
 'n03179701']

In [9]:
train_transforms = T.Compose([
    
  T.RandomResizedCrop(size, scale=(0.08,1), ratio=(3/4, 4/3), interpolation=T.InterpolationMode.BILINEAR), #crop random portion of image and resize it to given size.
  T.RandomHorizontalFlip(p=0.5), #p is probability that random image get flipped
  T.RandomAffine(degrees=(-15,15), fill=0, interpolation=T.InterpolationMode.BILINEAR), #randomly rotates
  T.RandomApply([T.ColorJitter(brightness=0.4, contrast=0.4,
                              saturation=0.4, hue=0.1)], p=0.8),    #randomly change brightness, contrast, saturation and hue
  T.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random'),
  T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

])

val_transforms = T.Compose([
    
    T.Resize(size, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(size),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

])

In [10]:
class TinyImagenetDataset(Dataset):
  def __init__(self, root, transforms, split, sub_dir):

    self.root = root
    self.sub_dir = sub_dir

    self.transforms = transforms

    self.full_path = os.path.join(root, split)
    self.Set = tuple()
    self.flatten = []

    self.class_names = None
    self.cls_to_idx = None
    self.idx_to_cls = None



    self.find_sort()

    self.make_set()

    if self.flatten == []:
      self.flatten_set()
      self.len = len(self.flatten)

  def find_sort(self):

    if not self.class_names:
      self.class_names = sorted([d for d in os.listdir(self.full_path) if os.path.isdir(os.path.join(self.full_path, d))])

    if not self.cls_to_idx:
      self.cls_to_idx = {cls:idx for idx, cls in enumerate(self.class_names)}

    if not self.idx_to_cls:
      self.idx_to_cls = {idx:cls for idx, cls in enumerate(self.class_names)}


  def make_set(self):

    for name in self.class_names:
      pair = ((name, self.cls_to_idx[name]),)
      self.Set += pair

  def flatten_set(self):


    for t in self.Set:

      cls_dir = os.path.join(self.full_path, t[0])
      img_dir = os.path.join(cls_dir, self.sub_dir)

      for img in os.listdir(img_dir):
        pair = (os.path.join(img_dir, img), t[1])
        self.flatten.append(pair)


  def process_img(self,idx):

    t = self.flatten[idx]


    img_uint8 = read_image(t[0])
    img = img_uint8.float().div(255.0)
    if img.shape[0] == 1:
      img = img.repeat(3,1,1)

    label = torch.tensor(t[1], dtype=torch.long)

    return img, label

  def __len__(self):

    return self.len

  def __getitem__(self, idx):

    img, label = self.process_img(idx)

    if self.transforms:
      img = self.transforms(img)

    return img,label

In [11]:
root = '/kaggle/working/tiny-net/tiny-imagenet-200'

In [12]:
class FormatFolder:
  def __init__(self, root, subdir):

    self.images = []
    self.labels = []

    self.dir = os.path.join(root, 'val')
    self.full_path = os.path.join(self.dir, subdir)
    self.extract()
    self.move_to_folder()

  def extract(self):
    with open(self.full_path) as f:
      for line in f:

        parts = line.strip().split('\t')
        self.images.append(parts[0])
        self.labels.append(parts[1])

  def move_to_folder(self):

    images_dir = os.path.join(self.dir, 'images')

    for img, label in zip(self.images, self.labels):

      cls_dir = os.path.join(self.dir, label)
      os.makedirs(cls_dir, exist_ok=True)

      source = os.path.join(images_dir, img)
      destination = os.path.join(cls_dir, img)

      if os.path.exists(source):
        shutil.move(source, destination)


In [13]:
FormatFolder(root, 'val_annotations.txt')
shutil.rmtree('/kaggle/working/tiny-net/tiny-imagenet-200/val/images', ignore_errors=True)

In [14]:
class PatchEmbedding(nn.Module):
  def __init__(self, patch_size, d_model):
    super().__init__()

    self.conv = nn.Conv2d(in_channels=3, out_channels=d_model, kernel_size=patch_size, stride=patch_size)
    self.flatten = nn.Flatten(start_dim=2)

    self.cls = nn.Parameter(torch.randn(1, 1, d_model))
  def forward(self, x):

    B = x.shape[0]
    patches = self.conv(x)
    patches = self.flatten(patches)

    patches = patches.transpose(1,2)

    cls_tokens = self.cls.expand(B, -1, -1)
    patches = torch.concat((cls_tokens, patches), dim=1)

    return patches

In [15]:
dummy_images = torch.randn(32, 3, 64, 64)

patch_embed = PatchEmbedding(patch_size=16, d_model=768)

output = patch_embed(dummy_images)

print(output.shape)

torch.Size([32, 17, 768])


In [16]:
class PositionalEncoding(nn.Module):
  def __init__(self, num_patches, d_model):
    super().__init__()

    pe = torch.zeros((num_patches+1, d_model))

    position = torch.arange(0,num_patches+1, dtype=torch.float).unsqueeze(1)

    div_term = torch.exp(
        -torch.arange(0, d_model, 2) * math.log(10000.0) / d_model
    )

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    pe = pe.unsqueeze(0)

    self.register_buffer('pe', pe)

  def forward(self, x):
    T = x.shape[1]

    return x + self.pe[:, :T]


In [17]:
# class Attention(nn.Module):
#   def __init__(self, d_model, d_k):

#     super().__init__()
#     self.d_k = d_k
#     self.WQ = nn.Linear(d_model, d_k, bias=False)
#     self.WK = nn.Linear(d_model, d_k, bias=False)
#     self.WV = nn.Linear(d_model, d_k, bias=False)

#     self.dropout = nn.Dropout(dropout)

#   def forward(self, x):

#     Q = self.WQ(x)
#     K = self.WK(x)
#     V = self.WV(x)

#     score = torch.matmul(Q, K.transpose(-2,-1)) * (self.d_k)**-0.5
#     score = F.softmax(score, dim=-1)
#     score = self.dropout(score)

#     out = torch.matmul(score, V)

#     return out

In [ ]:
class MHA(nn.Module):
  def __init__(self, d_model, n_heads, dropout=0.3):

    super().__init__()
    self.d_model = d_model
    self.n_heads = n_heads
    self.d_k = d_model // n_heads

    self.WQ = nn.Linear(d_model, d_model, bias=False)
    self.WK = nn.Linear(d_model, d_model, bias=False)
    self.WV = nn.Linear(d_model, d_model, bias=False)

    self.proj = nn.Linear(d_model, d_model, bias=False)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, _ = x.shape

    Q = self.WQ(x).view(B,T, self.n_heads, self.d_k).transpose(1,2)
    K = self.WK(x).view(B,T, self.n_heads, self.d_k).transpose(1,2)
    V = self.WV(x).view(B,T, self.n_heads, self.d_k).transpose(1,2)

    score = torch.matmul(Q, K.transpose(-2, -1)) * (self.d_k ** -0.5)
    attn = F.softmax(score, dim=-1)
    attn = self.dropout(attn)

    out = torch.matmul(attn, V)
    out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)

    return self.proj(out)

In [ ]:
class Encoder(nn.Module):
  def __init__(self,d_model, n_heads, dropout=0.3):
    super().__init__()

    self.mha = MHA(d_model, n_heads)
    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.mlp = nn.Sequential(*[
        nn.Linear(d_model, d_model*4),
        nn.GELU(),
        nn.Linear(d_model*4, d_model),
        nn.Dropout(dropout),
    ])

  def forward(self, x):

    x = x + self.mha(self.ln1(x))
    x = x + self.mlp(self.ln2(x))

    return x

In [ ]:
class ViT(nn.Module):
  def __init__(self, d_model, n_heads, num_classes, size=64, patch=16, n_layers=12, dropout=0.3):

    super().__init__()

    num_patches = (size // patch) ** 2

    self.patched_embd = PatchEmbedding(patch, d_model)
    self.pos_encd = PositionalEncoding(num_patches, d_model)

    self.encoders = nn.ModuleList([
        Encoder(d_model, n_heads, dropout) for _ in range(n_layers)
    ])

    self.ln = nn.LayerNorm(d_model)
    self.head = nn.Linear(d_model, num_classes)

  def forward(self, x):

    x = self.patched_embd(x)
    x = self.pos_encd(x)

    for encoder in self.encoders:
      x = encoder(x)

    cls_token = x[:, 0, :]

    out = self.ln(cls_token)
    out = self.head(out)

    return out

In [ ]:
dum_img = torch.rand((32, 3, 64, 64))
vit = ViT(768, 12, 200, size=64, patch=16, n_layers=12, dropout=0.3)

out = vit(dum_img)
out

tensor([[-0.2491, -0.2806, -0.5505,  ...,  0.8629, -0.0187, -0.7082],
        [-0.0883, -0.2979, -0.1965,  ...,  1.0179,  0.2145, -0.8612],
        [ 0.0208, -0.1132, -0.2067,  ...,  1.1971,  0.2246, -0.6434],
        ...,
        [-0.3682,  0.0155, -0.2245,  ...,  0.8449,  0.1504, -0.4842],
        [ 0.2286, -0.1926,  0.2676,  ...,  0.8937,  0.1747, -0.3972],
        [-0.1205,  0.0185, -0.3234,  ...,  1.3563,  0.2001, -0.5476]],
       grad_fn=<AddmmBackward0>)

In [22]:
out.shape

torch.Size([32, 200])

In [23]:
print(torch.cuda.device_count())

2


In [ ]:
# Launching the distributed training script
!python train.py

In [ ]:
with open('history.json', 'r') as f:
    history = json.load(f)

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))


ax1.plot(epochs_range, history['train_loss'], label='Train Loss', color='blue', marker='o', markersize=4)
ax1.plot(epochs_range, history['val_loss'], label='Val Loss', color='red', marker='x', markersize=4)
ax1.set_title("Training and Validation Loss") 
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(epochs_range, history['train_acc'], label='Train Acc', color='blue', marker='o', markersize=4)
ax2.plot(epochs_range, history['val_acc'], label='Val Acc', color='red', marker='x', markersize=4)
ax2.set_title("Training and Validation Accuracy") 
ax2.set_xlabel("Epochs")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()